In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.special import expit

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)

In [ ]:
TAG = 'arw4'
GENERATED_FROM = 1

In [ ]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [ ]:
initial = pd.read_csv(INPUT / TAG / 'initialization.csv').iloc[0]

In [ ]:
sim[0] = rng.exponential(initial.scale, N)
for row in params.itertuples():
    for t in range(row.start, row.end+1):
        loc = row.mode_beta * sim[t]
        lower = stats.laplace.cdf(0, loc=loc, scale=row.alpha)
        u = np.minimum(lower + (1-lower)*rng.random(N), np.nextafter(1.,0.))
        sim[t+1] = stats.laplace.ppf(u, loc=loc, scale=row.alpha)

In [ ]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))